# Treino do detector de caracteres (YOLO11n, 36 classes) no Kaggle

Antes de rodar, confira (aba direita do notebook):
1. **Data -> + Add Input** -> dois datasets: um com `data/yolo/` (imagens, labels e `data.yaml` ja prontos, gerados localmente por `build_dataset.py`) e outro com `src/` + `configs/train.yaml`.
2. **Settings -> Accelerator** -> `GPU T4 x2` (ou `P100`).
3. **Settings -> Internet** -> `On` (necessario para baixar o peso pre-treinado `yolo11n.pt`).

Este notebook so executa treino - teste rapido (smoke test) e treino completo, incluindo retomada em caso de interrupcao. Nenhuma etapa de preparacao de dados ou de avaliacao roda aqui.

In [ ]:
!pip install -q ultralytics pyyaml
import torch
print("CUDA disponivel:", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

## Localizar o dataset ja pronto (`data/yolo/`)

Procura por `data.yaml` em `/kaggle/input/` e usa a pasta onde ele esta como `data/yolo/` - nao assume um nome fixo de dataset, entao funciona com qualquer nome dado ao Kaggle Dataset no upload. Em seguida confere que a quantidade de imagens/labels por split bate com o esperado (protege contra um upload incompleto ou incompativel passar despercebido).

In [ ]:
import glob
from pathlib import Path

REPO_ROOT = Path("/kaggle/working/CharacterDetector")
(REPO_ROOT / "data").mkdir(parents=True, exist_ok=True)

yaml_matches = glob.glob("/kaggle/input/**/data.yaml", recursive=True)
assert yaml_matches, (
    "data.yaml nao encontrado em /kaggle/input - confira se o dataset com "
    "data/yolo/ (images/, labels/, data.yaml) foi adicionado (aba Data, + Add Input)."
)
yolo_dir = Path(yaml_matches[0]).resolve().parent
print("data/yolo encontrado em:", yolo_dir)

yolo_link = REPO_ROOT / "data" / "yolo"
if not yolo_link.exists():
    yolo_link.symlink_to(yolo_dir)
print("data/yolo ->", yolo_link.resolve())

# contagens do split gerado localmente (seed fixa em build_splits.py) - um
# numero diferente aqui indica upload incompleto ou um dataset de uma
# geracao de split diferente da esperada.
EXPECTED_COUNTS = {"train": 16565, "val": 3550, "test": 3549}
for split, expected in EXPECTED_COUNTS.items():
    n_images = len(list((yolo_link / "images" / split).glob("*.jpg")))
    n_labels = len(list((yolo_link / "labels" / split).glob("*.txt")))
    assert n_images == expected, f"esperava {expected} imagens em images/{split}, achei {n_images}"
    assert n_labels == expected, f"esperava {expected} labels em labels/{split}, achei {n_labels}"
    print(f"{split}: {n_images} imagens, {n_labels} labels - OK")

## Localizar e copiar o codigo (`src/` + `configs/`)

Procura por `training/train.py` em `/kaggle/input/` e copia `src/` e `configs/` (sem alterar nada dentro deles) para a pasta de trabalho desta sessao.

In [ ]:
import shutil

code_matches = glob.glob("/kaggle/input/**/training/train.py", recursive=True)
assert code_matches, (
    "train.py nao encontrado em /kaggle/input - confira se o dataset com src/ + "
    "configs/train.yaml foi adicionado (aba Data, + Add Input)."
)
src_dir = Path(code_matches[0]).resolve().parents[1]  # .../src
project_dir = src_dir.parent
configs_dir = project_dir / "configs"
assert configs_dir.exists(), f"esperava configs/ ao lado de src/ em {project_dir}, nao achei"

for name, srcp in (("src", src_dir), ("configs", configs_dir)):
    dst = REPO_ROOT / name
    if dst.exists():
        shutil.rmtree(dst)
    shutil.copytree(srcp, dst)

print("codigo copiado para", REPO_ROOT)

## Checagem rapida do `data.yaml`

Confirma que o arquivo copiado junto com o dataset tem as 36 classes esperadas antes de gastar tempo de GPU em cima dele.

In [ ]:
import yaml

with open(REPO_ROOT / "data" / "yolo" / "data.yaml", encoding="utf-8") as f:
    data_cfg = yaml.safe_load(f)

assert data_cfg["nc"] == 36, f"esperava nc=36, achei {data_cfg['nc']}"
assert len(data_cfg["names"]) == 36, f"esperava 36 nomes de classe, achei {len(data_cfg['names'])}"
print("nc:", data_cfg["nc"])
print("names:", data_cfg["names"])
print("OK, data.yaml consistente.")

## Detectar treino interrompido para retomar (opcional)

Procura em `/kaggle/input/` por um checkpoint `weights/last.pt` cujo `args.yaml` esteja presente na mesma pasta do run (sem o `args.yaml` ao lado, o Ultralytics nao consegue retomar corretamente - so o `.pt` sozinho nao basta). So considera um checkpoint cujo nome de pasta do run bate com o `name` configurado em `configs/train.yaml`, para nao confundir com um checkpoint de smoke test.

Se encontrar, copia a pasta do run inteira para um local gravavel desta sessao e a celula de treino completo, mais abaixo, usa `--resume` automaticamente. Se nao encontrar nada, o treino roda do zero normalmente.

Para usar isto depois de uma sessao interrompida: baixe a pasta inteira do run (por exemplo `outputs/runs/detect/full_run/`, com `args.yaml` e `weights/last.pt`) pelo painel de arquivos da sessao, suba como um novo Kaggle Dataset (ou adicione ao dataset de codigo), adicione como Input nesta nova sessao, e rode o notebook de novo do inicio.

In [ ]:
import yaml

with open(REPO_ROOT / "configs" / "train.yaml", encoding="utf-8") as f:
    configured_run_name = yaml.safe_load(f)["name"]

resume_checkpoint = None
for last_pt in glob.glob("/kaggle/input/**/weights/last.pt", recursive=True):
    run_dir = Path(last_pt).resolve().parents[1]
    if run_dir.name != configured_run_name:
        continue
    if not (run_dir / "args.yaml").exists():
        continue
    writable_run_dir = REPO_ROOT / "outputs" / "runs" / "detect" / run_dir.name
    writable_run_dir.parent.mkdir(parents=True, exist_ok=True)
    if writable_run_dir.exists():
        shutil.rmtree(writable_run_dir)
    shutil.copytree(run_dir, writable_run_dir)
    resume_checkpoint = writable_run_dir / "weights" / "last.pt"
    print(f"Checkpoint de retomada encontrado e copiado: {resume_checkpoint}")
    break

if resume_checkpoint is None:
    print("Nenhum checkpoint de retomada encontrado - o treino completo vai comecar do zero.")

## Teste rapido na GPU do Kaggle - NAO PULE ESTA ETAPA

Roda um treino pequeno (poucas epocas, subconjunto de imagens) antes de comprometer horas de cota no treino completo, usando o MESMO `--batch` de `configs/train.yaml` (nao um valor pequeno a parte) para essa etapa ja validar de verdade o que o treino completo vai usar. Confirma tres coisas ao mesmo tempo:

1. O pipeline roda de ponta a ponta nesta sessao do Kaggle e produz um `.pt` de verdade.
2. O batch configurado cabe na VRAM disponivel - e, com mais de uma GPU visivel (ex.: "GPU T4 x2"), que o treino distribuido (DDP) entre elas sobe sem erro.
3. Os checkpoints sao gravados a cada epoca - enquanto a celula abaixo ainda estiver rodando, abra o painel de arquivos da sessao e confira que `outputs/runs/detect/smoke_test/weights/last.pt` e `/kaggle/working/checkpoints/last.pt` aparecem e sao atualizados a cada epoca.

In [ ]:
import yaml

with open(REPO_ROOT / "configs" / "train.yaml", encoding="utf-8") as f:
    real_batch = yaml.safe_load(f)["batch"]

smoke_cmd = f"python {REPO_ROOT}/src/training/train.py --smoke-test --n 300 --smoke-epochs 5 --batch {real_batch} --live-copy-dir /kaggle/working/checkpoints"
print("Executando:", smoke_cmd)
!{smoke_cmd}

## Treino completo

Le os hiperparametros de `configs/train.yaml` (`imgsz`, `epochs`, `batch`, `patience`, `save_period`) e usa automaticamente todas as GPUs visiveis via DDP quando ha mais de uma (o `batch` do config e o TOTAL, dividido entre elas). Se a celula de deteccao de retomada (acima) encontrou um checkpoint de uma sessao anterior, continua o treino a partir dele automaticamente; caso contrario, comeca do zero a partir do peso pre-treinado `yolo11n.pt` (baixado automaticamente na primeira execucao, ja que **Internet** esta `On`).

`--live-copy-dir` copia `last.pt`/`best.pt` para `/kaggle/working/checkpoints` a cada epoca, alem do local padrao em `outputs/runs/detect/<nome-do-run>/weights/`.

Prefira **Save Version -> Save & Run All (Commit)** em vez de rodar celula por celula com a aba aberta, para nao depender da conexao do navegador ficar aberta. Mesmo assim, volte periodicamente durante o treino e baixe pelo painel de arquivos da sessao (funciona com a sessao ainda rodando, diferente da aba Output que so empacota o resultado no final):
- `/kaggle/working/checkpoints/last.pt` - rapido de conferir.
- a pasta inteira `outputs/runs/detect/<nome-do-run>/` - precisa dela completa, com `args.yaml` ao lado de `weights/`, para poder retomar depois; so o `.pt` nao e suficiente.

O que estiver salvo em disco no momento em que a sessao cair e o que sobra - baixar periodicamente por conta propria e a unica protecao real contra perder o progresso todo.

In [ ]:
train_cmd = f"python {REPO_ROOT}/src/training/train.py --live-copy-dir /kaggle/working/checkpoints"
if resume_checkpoint is not None:
    train_cmd = f"python {REPO_ROOT}/src/training/train.py --resume {resume_checkpoint} --live-copy-dir /kaggle/working/checkpoints"

print("Executando:", train_cmd)
!{train_cmd}

## Deixar o modelo final facil de baixar

Copia o resultado para a raiz de `/kaggle/working/`: usa o `best.pt` do run se o treino terminou normalmente, ou cai para o `last.pt` mais recente em `--live-copy-dir` se a sessao foi interrompida no meio.

In [ ]:
best = REPO_ROOT / "outputs" / "runs" / "detect" / configured_run_name / "weights" / "best.pt"
live_copy_last = Path("/kaggle/working/checkpoints/last.pt")

if best.exists():
    shutil.copy2(best, "/kaggle/working/best.pt")
    print("Treino completo: /kaggle/working/best.pt (best.pt do run) - baixe pela aba Output/Data.")
elif live_copy_last.exists():
    shutil.copy2(live_copy_last, "/kaggle/working/best.pt")
    print(
        "[aviso] best.pt final nao encontrado (treino pode ter sido interrompido) - "
        "usando o ultimo checkpoint salvo em --live-copy-dir em vez disso."
    )
    print("/kaggle/working/best.pt (na verdade e o last.pt mais recente) - baixe pela aba Output/Data.")
else:
    raise RuntimeError(
        "nao encontrei nem o best.pt do run nem nada em --live-copy-dir - confira os logs do treino acima."
    )